# AyuConnect · Gemma 4 on-device medical intake

**Submission for the [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon) · Health & Sciences track.**

AyuConnect is a privacy-first, offline-capable medical intake assistant. A patient describes their symptoms by text, voice, or photo; Gemma 4 conducts a focused intake; the model calls deterministic tools (triage, drug-interaction, ICD-10 lookup) to ground its reasoning; and a clinician receives a structured handoff. **Nothing leaves the device.**

This notebook reproduces the end-to-end pipeline that powers the public demo at https://github.com/VineethRV/AyuConnect.

What you'll see, in order:

1. Load **`google/gemma-4-E4B-it`** — the 4.5B multimodal instruction-tuned variant designed for on-device use.
2. Run a **text-only intake** with native function calling against our offline medical knowledge base.
3. Run an **image-grounded intake** — patient sends a photo of a rash; Gemma 4 sees it and adapts.
4. Run an **audio intake** — patient speaks symptoms in a voice note; Gemma 4 transcribes and reasons over them.
5. Show the **doctor-side consultation note** drafted from the same transcript.

## Why Gemma 4?

| Capability | What AyuConnect uses it for |
|---|---|
| E4B (4.5B effective params) | Runs on a clinic laptop without a GPU — fits the "no-cloud-required" brief. |
| Text + image + audio + video | Patients with low literacy can speak; rural workers can photograph the wound. |
| Native function calling | The model reaches for deterministic tools (triage, drug-interaction) instead of hallucinating. |
| 128k context | The whole intake transcript plus the local KB fits in one context. |
| Apache 2.0 weights | Clinics can deploy without per-call royalties. |

## 0. Install dependencies

On Kaggle's GPU runtime, `transformers` and `torch` are pre-installed. We just refresh `transformers` to a version that supports Gemma 4's multimodal `apply_chat_template`.

In [ ]:
%pip install -q --upgrade "transformers>=4.46.0" accelerate Pillow soundfile
%pip install -q httpx pydantic

In [ ]:
# Make the AyuConnect backend package importable from the notebook.
# On Kaggle, clone the repo as input dataset; here we add it to sys.path.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd() / '..' / 'backend').exists() else pathlib.Path('/kaggle/input/ayuconnect')
sys.path.insert(0, str(ROOT))
import os
os.environ['AYU_BACKEND'] = 'transformers'
os.environ['AYU_MODEL_ID'] = 'google/gemma-4-E4B-it'
print('Using AyuConnect at', ROOT)

## 1. Load Gemma 4 · E4B instruction-tuned

We pick E4B because it is the smallest variant that supports **all four modalities** (text/image/audio/video). At ~4.5B effective parameters it runs comfortably on a single consumer GPU and — with quantisation — on a high-end CPU laptop.

In [ ]:
from transformers import AutoProcessor
try:
    from transformers import AutoModelForMultimodalLM as _Model
except ImportError:
    from transformers import AutoModelForCausalLM as _Model
import torch

MODEL_ID = 'google/gemma-4-E4B-it'
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = _Model.from_pretrained(MODEL_ID, device_map='auto', torch_dtype=torch.bfloat16)
print('Loaded', MODEL_ID, 'on', next(model.parameters()).device)

## 2. Tool schemas (offline) · the grounding layer

Every tool is a pure function over a small JSON medical knowledge base shipped with the repo — no network calls. That is what lets AyuConnect run on a clinic LAN with no internet.

We reuse the production tool registry, so what the notebook demonstrates is exactly what the deployed app does.

In [ ]:
from backend.app.tools import tool_schemas, invoke_tool, TOOLS
print(f'{len(TOOLS)} tools registered:')
for name, t in TOOLS.items():
    print(f'  • {name:32s} — {t.description[:72]}…')

## 3. The Gemma 4 tool-loop

This is the same orchestration the FastAPI backend runs — we lift it into the notebook so the reviewer can step through it. The model emits tool calls in its native JSON schema; we dispatch them, append results back to the conversation, and re-prompt until the model either replies in plain text or calls `submit_diagnosis_report`.

In [ ]:
from backend.app.prompts import INTAKE_SYSTEM, DOCTOR_REVIEW_SYSTEM
import json, uuid
from PIL import Image

def generate_with_tools(messages, tools, max_new_tokens=512):
    inputs = processor.apply_chat_template(
        messages, tools=tools, tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors='pt',
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    txt = processor.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    if hasattr(processor, 'parse_response'):
        parsed = processor.parse_response(txt)
        return parsed.get('content', '') or '', parsed.get('tool_calls', []) or []
    return txt.strip(), []

def run_intake(user_messages, max_loops=5):
    msgs = [{'role': 'system', 'content': INTAKE_SYSTEM}, *user_messages]
    tools = tool_schemas()
    trace = []
    for _ in range(max_loops):
        content, tool_calls = generate_with_tools(msgs, tools)
        trace.append({'assistant_text': content, 'tool_calls': tool_calls})
        if not tool_calls:
            return content, trace
        msgs.append({'role': 'assistant', 'content': content,
                     'tool_calls': [{'id': f'c{i}', 'type': 'function',
                                     'function': {'name': tc['name'], 'arguments': tc.get('arguments', {})}}
                                    for i, tc in enumerate(tool_calls)]})
        for i, tc in enumerate(tool_calls):
            result = invoke_tool(tc['name'], tc.get('arguments', {}))
            msgs.append({'role': 'tool', 'tool_call_id': f'c{i}', 'name': tc['name'],
                         'content': json.dumps(result, ensure_ascii=False)})
            trace[-1].setdefault('tool_results', []).append(result)
    return '(budget exceeded)', trace

## 4. Demo · text intake with red-flag triage

A 58-year-old patient reports crushing chest pain. Gemma 4 should:
1. Call `triage_severity` immediately.
2. See `urgency=red` and call `submit_diagnosis_report` without continuing the interview.
3. Tell the patient to seek emergency care.

In [ ]:
reply, trace = run_intake([
    {'role': 'system', 'content': 'Patient meta: age=58, sex=Male.'},
    {'role': 'user', 'content': 'I have crushing chest pain spreading to my left arm and I\'m sweating heavily.'},
])
print('ASSISTANT:', reply, '\n')
for step in trace:
    if step['tool_calls']:
        for tc, res in zip(step['tool_calls'], step.get('tool_results', [])):
            print(f"\u21b3 tool {tc['name']}({tc.get('arguments')}) -> {res}")

## 5. Demo · image-grounded intake

A patient sends a photograph of a skin lesion along with a brief description. Gemma 4's vision encoder sees the image; the model describes the morphology and refines its differential.

In [ ]:
import urllib.request, io
url = 'https://upload.wikimedia.org/wikipedia/commons/3/35/Atopic_dermatitis_child.jpg'
with urllib.request.urlopen(url) as r:
    img = Image.open(io.BytesIO(r.read())).convert('RGB')

reply, trace = run_intake([
    {'role': 'system', 'content': 'Patient meta: age=8, sex=Female.'},
    {'role': 'user', 'content': [
        {'type': 'image', 'image': img},
        {'type': 'text', 'text': "This is my child's elbow. She scratches it constantly, especially at night, and the skin gets red and flaky."},
    ]},
])
print('ASSISTANT:', reply, '\n')
for step in trace:
    for tc in step.get('tool_calls', []) or []:
        print(f"\u21b3 tool {tc['name']}({tc.get('arguments')})")

## 6. Demo · audio intake (low-literacy / accessibility)

Many patients in low-resource settings cannot type fluently. With Gemma 4's audio encoder we let them **speak** their symptoms; the model understands the audio directly — no separate ASR model required.

We use a tiny synthetic clip here for reproducibility; in the deployed app the browser records via `MediaRecorder`.

In [ ]:
import numpy as np, soundfile as sf
# Replace with any short WAV your reviewer wants to try.
demo_clip = '/kaggle/input/sample-audio/symptom_voice_note.wav'
if not pathlib.Path(demo_clip).exists():
    # Fallback: synthesise silence so the cell still runs end-to-end.
    demo_clip = '/tmp/silence.wav'
    sf.write(demo_clip, np.zeros(16000), 16000)

audio, sr = sf.read(demo_clip)
reply, trace = run_intake([
    {'role': 'user', 'content': [
        {'type': 'audio', 'audio': audio, 'sampling_rate': sr},
        {'type': 'text', 'text': 'Please listen to this voice note from a patient and start the intake.'},
    ]},
])
print('ASSISTANT:', reply)

## 7. Demo · doctor consultation note

Once the patient intake is finalised, the doctor opens the case in the clinician portal. The same Gemma 4 model — with a different system prompt — drafts a SOAP note and proactively calls `check_drug_interactions` against the patient's current medications.

In [ ]:
doctor_msgs = [
    {'role': 'system', 'content': DOCTOR_REVIEW_SYSTEM},
    {'role': 'user', 'content': (
        'Patient John Doe, 45M. Intake-suggested top conditions: Type 2 Diabetes, Hypertension, Asthma. '
        'Symptoms: fatigue, shortness of breath, increased thirst. '
        'Current medications: metformin, lisinopril, ibuprofen, salbutamol. '
        'Please draft a SOAP note and flag any drug interactions.'
    )},
]
reply, trace = run_intake([{'role': 'user', 'content': doctor_msgs[1]['content']}])
print(reply)

## 8. Where to go next

* **Source code & deployment instructions** — https://github.com/VineethRV/AyuConnect
* **Public demo video** — see `docs/video.md` in the repo for the script.
* **Run on a laptop** — `docker compose up` brings up the Ollama backend + Next.js frontend.
* **Switch backends** — a single env var (`AYU_BACKEND={ollama,transformers,vllm}`) selects how Gemma 4 is served.

Built for the Health & Sciences track of the Gemma 4 Good Hackathon · May 2026.